In [2]:
from pathlib import Path 
import json 
import pandas as pd 

In [3]:
RAW_DIR = Path("../../data/raw")
spotify_files = list(RAW_DIR.rglob("*.json"))

spotify_files

[PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Audio_2026.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Audio_2024.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Audio_2025_1.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Audio_2023.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Audio_2024_1.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Audio_2025_2.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Video_2026.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Video_2025.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Audio_2026_1.json'),
 PosixPath('../../data/raw/Spotify Extended Streaming History/Streaming_History_Video_2024.json'),
 P

In [4]:
for path in spotify_files:
    print(Path)

<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>
<class 'pathlib.Path'>


In [5]:
history_files = [
    path for path in spotify_files
    if "Streaming" in path.name or "Audio" in path.name
]

rows = []

for path in history_files:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        rows.extend(data)

history_df = pd.DataFrame(rows)

history_df.shape

(111521, 23)

In [6]:
history_df.columns.tolist()

['ts',
 'platform',
 'ms_played',
 'conn_country',
 'ip_addr',
 'master_metadata_track_name',
 'master_metadata_album_artist_name',
 'master_metadata_album_album_name',
 'spotify_track_uri',
 'episode_name',
 'episode_show_name',
 'spotify_episode_uri',
 'audiobook_title',
 'audiobook_uri',
 'audiobook_chapter_uri',
 'audiobook_chapter_title',
 'reason_start',
 'reason_end',
 'shuffle',
 'skipped',
 'offline',
 'offline_timestamp',
 'incognito_mode']

In [7]:
history_df.head()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2026-01-01T00:40:03Z,ios,32852,BE,2a02:a03f:66a2:3500:5019:9dba:7fd1:3528,time alone w u,Artemas,time alone w u,spotify:track:5l8XXqPtwQQ28ThnjaDUh5,NaN,...,NaN,NaN,NaN,trackdone,unexpected-exit-while-paused,True,False,False,1.767214e+09,False
1,2026-01-01T00:40:34Z,ios,5082,BE,2a02:a03f:66a2:3500:5019:9dba:7fd1:3528,time alone w u,Artemas,time alone w u,spotify:track:5l8XXqPtwQQ28ThnjaDUh5,NaN,...,NaN,NaN,NaN,appload,endplay,True,True,False,1.767228e+09,False
2,2026-01-01T00:40:36Z,ios,2090,BE,2a02:a03f:66a2:3500:5019:9dba:7fd1:3528,Stateside + Zara Larsson,PinkPantheress,Fancy Some More?,spotify:track:1DwscornXpj8fmOmYVlqZt,NaN,...,NaN,NaN,NaN,clickrow,endplay,True,True,False,1.767228e+09,False
3,2026-01-01T00:40:36Z,ios,464,BE,2a02:a03f:66a2:3500:5019:9dba:7fd1:3528,Moth To A Flame (with The Weeknd),Swedish House Mafia,Paradise Again,spotify:track:0VO8gYVDSwM1Qdd2GsMoYK,NaN,...,NaN,NaN,NaN,clickrow,endplay,True,True,False,1.767228e+09,False
4,2026-01-01T00:40:37Z,ios,789,BE,2a02:a03f:66a2:3500:5019:9dba:7fd1:3528,Teacher's Pet,Melanie Martinez,K-12,spotify:track:5o5akY9xnEk6lpMkD8RwD9,NaN,...,NaN,NaN,NaN,clickrow,endplay,True,True,False,1.767228e+09,False


In [8]:
history_df["hours_played"] = history_df["ms_played"] / 1000 / 60 / 60

top_tracks = (
    history_df
    .groupby(["master_metadata_album_artist_name", "master_metadata_track_name"], dropna=False)
    .agg(
        plays=("ms_played", "count"),
        hours_played=("hours_played", "sum"),
        total_ms=("ms_played", "sum"),
    )
    .reset_index()
    .sort_values("hours_played", ascending=False)
)

top_tracks.head(30)

,master_metadata_album_artist_name,master_metadata_track_name,plays,hours_played,total_ms
1892,Billie Eilish,WILDFLOWER,556,27.420198,98712714
11403,Tate McRae,It's ok I'm ok,872,27.161526,97781494
6972,Lah Pat,Rodeo (Remix),439,22.871526,82337494
11188,Swedish House Mafia,Moth To A Flame (with The Weeknd),543,22.759139,81932899
741,Amaarae,SAD GIRLZ LUV MONEY Remix (feat. Kali Uchis an...,395,18.379973,66167901
2724,Chquan,Angel Numbers Amapiano - Remix,385,17.149470,61738093
11771,The Weeknd,"One Of The Girls (with JENNIE, Lily Rose Depp)",407,16.248459,58494453
5882,Jay Sean,Ride It,500,16.147363,58130508
3129,DENNIS,Tá OK (Remix),355,15.969989,57491960
2884,Cigarettes After Sex,Cry,263,15.560192,56016690


In [12]:
history_df["ts"] = pd.to_datetime(history_df["ts"], errors="coerce")

history_clean = history_df[
    history_df["master_metadata_track_name"].notna()
].copy()

history_clean.shape

(111278, 24)

In [13]:
BTS_ARTISTS = [
    "BTS",
    "RM",
    "Jin",
    "SUGA",
    "Agust D",
    "j-hope",
    "Jimin",
    "V",
    "Jung Kook",
    "Jungkook",
]

bts_history = history_clean[
    history_clean["master_metadata_album_artist_name"].isin(BTS_ARTISTS)
].copy()

bts_history.shape

(2855, 24)

In [14]:
bts_song_league = (
    bts_history
    .groupby(
        [
            "master_metadata_album_artist_name",
            "master_metadata_track_name",
            "master_metadata_album_album_name",
        ],
        dropna=False,
    )
    .agg(
        plays=("ms_played", "count"),
        hours_played=("hours_played", "sum"),
        total_ms=("ms_played", "sum"),
        first_play=("ts", "min"),
        last_play=("ts", "max"),
    )
    .reset_index()
    .sort_values("hours_played", ascending=False)
)

bts_song_league.head(30)

,master_metadata_album_artist_name,master_metadata_track_name,master_metadata_album_album_name,plays,hours_played,total_ms,first_play,last_play
264,BTS,Tomorrow,Skool Luv Affair,101,3.836258,13810530,2026-01-24 15:25:52+00:00,2026-07-08 20:53:29+00:00
162,BTS,Let Me Know,Dark & Wild,50,2.312484,8324941,2026-07-04 06:59:47+00:00,2026-07-08 19:21:08+00:00
179,BTS,MIC Drop (Steve Aoki Remix) (Full Length Edition),Love Yourself 結 'Answer',54,2.123062,7643023,2026-01-24 14:51:12+00:00,2026-07-08 18:14:53+00:00
218,BTS,Pied Piper,Love Yourself 承 'Her',55,1.935010,6966035,2024-07-24 18:45:38+00:00,2026-07-08 09:58:57+00:00
120,BTS,House of Cards (Full Length Edition),The Most Beautiful Moment in Life: Young Forever,39,1.917402,6902647,2024-11-02 12:28:14+00:00,2026-07-04 14:49:20+00:00
97,BTS,FAKE LOVE,Love Yourself 轉 'Tear',107,1.789293,6441454,2024-08-29 18:39:12+00:00,2026-07-07 16:05:24+00:00
200,BTS,Not Today,You Never Walk Alone,55,1.733658,6241168,2025-04-23 07:58:12+00:00,2026-07-08 20:03:19+00:00
22,BTS,2.0,ARIRANG,48,1.700099,6120356,2026-03-25 18:12:07+00:00,2026-07-08 21:46:04+00:00
309,Jung Kook,Stay Alive (Prod. SUGA of BTS),Stay Alive (Prod. SUGA of BTS),81,1.671669,6018009,2025-02-26 11:09:21+00:00,2026-07-08 18:18:36+00:00
195,BTS,My Time,MAP OF THE SOUL : 7,48,1.586619,5711829,2024-11-05 09:57:38+00:00,2026-07-08 18:15:04+00:00


In [15]:
bts_album_league = (
    bts_history
    .groupby(
        [
            "master_metadata_album_artist_name",
            "master_metadata_album_album_name",
        ],
        dropna=False,
    )
    .agg(
        plays=("ms_played", "count"),
        hours_played=("hours_played", "sum"),
        first_play=("ts", "min"),
        last_play=("ts", "max"),
    )
    .reset_index()
    .sort_values("hours_played", ascending=False)
)

bts_album_league.head(20)


,master_metadata_album_artist_name,master_metadata_album_album_name,plays,hours_played,first_play,last_play
6,BTS,ARIRANG,272,7.338642,2026-03-21 15:27:06+00:00,2026-07-08 21:46:04+00:00
30,BTS,MAP OF THE SOUL : 7,219,6.942204,2024-08-23 14:34:25+00:00,2026-07-08 19:23:30+00:00
29,BTS,Love Yourself 轉 'Tear',226,5.121804,2024-08-29 18:39:12+00:00,2026-07-08 09:42:23+00:00
27,BTS,Love Yourself 承 'Her',156,4.840806,2024-07-24 18:45:38+00:00,2026-07-08 15:34:46+00:00
42,BTS,Skool Luv Affair,130,4.596744,2024-11-05 10:08:55+00:00,2026-07-08 20:53:29+00:00
28,BTS,Love Yourself 結 'Answer',182,4.549797,2024-09-04 19:24:41+00:00,2026-07-08 19:21:48+00:00
49,BTS,Wings,211,4.130166,2024-09-05 04:25:05+00:00,2026-07-08 19:23:32+00:00
15,BTS,Dark & Wild,111,3.737432,2025-04-23 07:50:23+00:00,2026-07-08 19:21:08+00:00
47,BTS,The Most Beautiful Moment in Life: Young Forever,133,3.581411,2024-11-02 12:28:14+00:00,2026-07-05 17:15:19+00:00
46,BTS,The Most Beautiful Moment in Life Pt.2,98,2.719907,2024-08-23 14:40:49+00:00,2026-07-08 09:24:58+00:00


In [ ]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "../data" / "processed"

OUT_DIR = PROCESSED_DIR / "personal_atlas"
OUT_DIR.mkdir(parents=True, exist_ok=True)

history_clean.to_parquet(OUT_DIR / "spotify_history_clean.parquet", index=False)
bts_history.to_parquet(OUT_DIR / "bts_listening_history.parquet", index=False)
bts_song_league.to_parquet(OUT_DIR / "bts_song_league.parquet", index=False)
bts_album_league.to_parquet(OUT_DIR / "bts_album_league.parquet", index=False)